# 02 — Feature engineering: player clustering inputs

**Owner:** Rabin (W3)  
**Consumers:** Li (clustering / dashboard), Teem (dbt alignment)

## Goals
1. Build player-season per-90 stats from **raw StatsBomb tables** in the Databricks BQ catalog (same source as `05_player_clustering.py`, extended for all 11 features).
2. Validate all **11 clustering features** from `player_clustering.ipynb` are present.
3. Note gaps vs Teem's `int_player_season_stats` dbt model for later pipeline parity.

**Databricks:** keep `%python` / `%sql` as the first line of each code cell. Locally, remove those magic lines.

**Source tables:** `bq_raw_statsbomb_sa_catalog.raw_statsbomb.{events,lineups,matches}`

In [ ]:
%python
from pyspark.sql import functions as F

RAW = "bq_raw_statsbomb_sa_catalog.raw_statsbomb"
OUTPUT_CATALOG = "main"
OUTPUT_SCHEMA = "football_rq2"
OUT = f"{OUTPUT_CATALOG}.{OUTPUT_SCHEMA}"
MIN_MINUTES = 450  # minutes on pitch (duration sum), same as player_clustering.ipynb

CLUSTER_FEATURES = [
    "shots_p90",
    "xg_p90",
    "xg_per_shot",
    "dribbles_p90",
    "carries_att_third_p90",
    "passes_att_third_p90",
    "pass_completion_pct",
    "pressures_p90",
    "interceptions_p90",
    "clearances_p90",
    "duels_p90",
]

print(f"Clustering features: {len(CLUSTER_FEATURES)}")

## 1) Build player-season stats from raw events

Extends the `05_player_clustering` SQL with:
- `passes_att_third` / `carries_att_third` (`location_x > 80`)
- `pass_completion_pct`, `xg_per_shot`

Attacking-third proxy: passes/carries **starting** in the final third (StatsBomb open data has start `location_x` only).

In [ ]:
%sql
WITH player_season AS (
  SELECT
    e.player_id,
    l.player_name,
    l.position_name,
    m.competition_name,
    m.season_name,
    COUNT(DISTINCT e.match_id) AS matches_played,
    SUM(e.duration) AS total_minutes,
    SUM(e.duration) / 90.0 AS ninety_minutes,
    SUM(CASE WHEN e.type = 'Shot' THEN 1 ELSE 0 END) AS shots,
    SUM(CASE WHEN e.type = 'Shot' THEN COALESCE(e.shot_statsbomb_xg, 0) ELSE 0 END) AS xg,
    SUM(CASE WHEN e.type = 'Pass' THEN 1 ELSE 0 END) AS passes,
    SUM(CASE WHEN e.type = 'Pass' AND e.pass_outcome IS NULL THEN 1 ELSE 0 END) AS passes_completed,
    SUM(CASE WHEN e.type = 'Pass' AND e.location_x > 80 THEN 1 ELSE 0 END) AS passes_att_third,
    SUM(CASE WHEN e.type = 'Pressure' THEN 1 ELSE 0 END) AS pressures,
    SUM(CASE WHEN e.type = 'Carry' THEN 1 ELSE 0 END) AS carries,
    SUM(CASE WHEN e.type = 'Carry' AND e.location_x > 80 THEN 1 ELSE 0 END) AS carries_att_third,
    SUM(CASE WHEN e.type = 'Dribble' THEN 1 ELSE 0 END) AS dribbles,
    SUM(CASE WHEN e.type = 'Interception' THEN 1 ELSE 0 END) AS interceptions,
    SUM(CASE WHEN e.type = 'Clearance' THEN 1 ELSE 0 END) AS clearances,
    SUM(CASE WHEN e.type = 'Duel' THEN 1 ELSE 0 END) AS duels
  FROM bq_raw_statsbomb_sa_catalog.raw_statsbomb.events e
  JOIN bq_raw_statsbomb_sa_catalog.raw_statsbomb.lineups l
    ON e.player_id = l.player_id AND e.match_id = l.match_id
  JOIN bq_raw_statsbomb_sa_catalog.raw_statsbomb.matches m
    ON e.match_id = m.match_id
  WHERE e.player_id IS NOT NULL
    AND e.duration IS NOT NULL
  GROUP BY
    e.player_id,
    l.player_name,
    l.position_name,
    m.competition_name,
    m.season_name
  HAVING SUM(e.duration) >= 450
)
SELECT
  player_id,
  player_name,
  position_name,
  competition_name,
  season_name,
  matches_played,
  total_minutes,
  try_divide(shots, ninety_minutes) AS shots_p90,
  try_divide(xg, ninety_minutes) AS xg_p90,
  CASE WHEN shots > 0 THEN xg / shots ELSE 0 END AS xg_per_shot,
  try_divide(dribbles, ninety_minutes) AS dribbles_p90,
  try_divide(carries_att_third, ninety_minutes) AS carries_att_third_p90,
  try_divide(passes_att_third, ninety_minutes) AS passes_att_third_p90,
  CASE WHEN passes > 0 THEN 100.0 * passes_completed / passes ELSE NULL END AS pass_completion_pct,
  try_divide(pressures, ninety_minutes) AS pressures_p90,
  try_divide(interceptions, ninety_minutes) AS interceptions_p90,
  try_divide(clearances, ninety_minutes) AS clearances_p90,
  try_divide(duels, ninety_minutes) AS duels_p90
FROM player_season

In [ ]:
%python

player_season = _sqldf
player_season.createOrReplaceTempView("player_season_stats")

print("Rows:", player_season.count())
player_season.printSchema()
display(player_season.limit(10))

## 2) Validate all 11 clustering features

In [ ]:
%python
import pandas as pd

available = set(player_season.columns)
validation_rows = []
for feat in CLUSTER_FEATURES:
    present = feat in available
    null_pct = None
    if present:
        null_pct = float(
            player_season.agg(F.avg(F.col(feat).isNull().cast("double"))).collect()[0][0]
        ) * 100
    validation_rows.append(
        {
            "cluster_feature": feat,
            "present": present,
            "pct_null": round(null_pct, 2) if null_pct is not None else None,
            "source": "raw events SQL",
        }
    )

validation_df = pd.DataFrame(validation_rows)
display(validation_df)

missing = [f for f in CLUSTER_FEATURES if f not in available]
if missing:
    raise ValueError(f"Missing clustering features: {missing}")
print("All 11 clustering features present.")

id_cols = [
    "player_id",
    "player_name",
    "position_name",
    "competition_name",
    "season_name",
    "matches_played",
    "total_minutes",
]
clustering_features = player_season.select(*id_cols, *CLUSTER_FEATURES)
clustering_features.createOrReplaceTempView("player_clustering_features")

null_exprs = [
    F.round(100 * F.avg(F.col(c).isNull().cast("double")), 2).alias(f"pct_null_{c}")
    for c in CLUSTER_FEATURES
]
display(clustering_features.agg(*null_exprs))

## 3) Findings for Li & Teem

### From raw StatsBomb (this notebook)
All **11/11** clustering features are built directly — no dbt dependency, no proxies.

| Feature | Raw events logic |
|---|---|
| `shots_p90`, `xg_p90` | `Shot` count / xG sum ÷ ninety_minutes |
| `xg_per_shot` | `xg / shots` (0 if no shots) |
| `dribbles_p90` | `Dribble` |
| `carries_att_third_p90` | `Carry` with `location_x > 80` |
| `passes_att_third_p90` | `Pass` with `location_x > 80` |
| `pass_completion_pct` | `passes_completed / passes` (null if 0 passes) |
| `pressures_p90` | `Pressure` |
| `interceptions_p90` | `Interception` |
| `clearances_p90` | `Clearance` |
| `duels_p90` | `Duel` |

**Li:** use `player_clustering_features` (or saved Delta table below) — ready for PCA + K-Means.

**Teem:** `int_player_season_stats` dbt model still lacks att-third metrics, dribbles, clearances, and full duels — mirror this SQL in `int_player_match_stats` → season rollup when convenient.

In [ ]:
%python

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {OUTPUT_CATALOG}.{OUTPUT_SCHEMA}")

clustering_features.write.mode("overwrite").format("delta").saveAsTable(
    f"{OUT}.player_clustering_features"
)

validation_sdf = spark.createDataFrame(validation_df)
validation_sdf.write.mode("overwrite").format("delta").saveAsTable(
    f"{OUT}.clustering_feature_validation"
)

print("Saved:")
print(f"  {OUT}.player_clustering_features")
print(f"  {OUT}.clustering_feature_validation")